In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, Conv2D, Flatten, MaxPooling2D
from results_utils import DATA_DIR, callbacks_paths, save_results
import matplotlib.pyplot as plt


tf.keras.utils.set_random_seed(42)
MODEL_NAME = "cnn_model_1"

In [ ]:
import tensorflow as tf

def build_cnn_model():
    cnn_model = tf.keras.Sequential([
        # Entrée + Normalisation [0, 255] -> [0, 1]
        tf.keras.layers.Input(shape=(224, 224, 3)),
        tf.keras.layers.Rescaling(1./255),

        # Bloc 1
        tf.keras.layers.Conv2D(filters=32, kernel_size=(3,3), padding='same', kernel_initializer='he_uniform', activation=tf.nn.relu),
        tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=2),

        # Bloc 2
        tf.keras.layers.Conv2D(filters=64, kernel_size=(3,3), padding='same', kernel_initializer='he_uniform', activation=tf.nn.relu),
        tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=2),

        # Bloc 3 (ajouté pour réduire la taille et capter des formes plus complexes)
        tf.keras.layers.Conv2D(filters=128, kernel_size=(3,3), padding='same', kernel_initializer='he_uniform', activation=tf.nn.relu),
        tf.keras.layers.MaxPool2D(pool_size=(2,2), strides=2),

        # Classification (GlobalAveragePooling2D évite les 24 millions de paramètres)
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation=tf.nn.relu, kernel_initializer='he_uniform'),
        tf.keras.layers.Dense(10, activation=tf.nn.softmax)
    ])

    return cnn_model

cnn_model = build_cnn_model()
cnn_model.summary()

In [ ]:
cnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "valid",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

best_model_path, history_path = callbacks_paths(MODEL_NAME)

history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop,
               tf.keras.callbacks.ModelCheckpoint(best_model_path, monitor="val_loss", save_best_only=True),
               tf.keras.callbacks.CSVLogger(history_path)]
)

In [ ]:
test_loss, test_acc = cnn_model.evaluate(test_ds)
print(f"Précision finale sur le test : {test_acc:.2%}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Précision')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Perte (Loss)')
plt.show()

In [ ]:
params = {"epochs": 15, "lr": 1e-3, "patience": 3, "batch_size": BATCH_SIZE,
          "filters": "(32, 64, 128)", "dropout": 0.4, "dense_units": 128,
          "augment": False, "batchnorm": False, "class_weight": False}

save_results(MODEL_NAME, cnn_model, test_ds, test_ds.class_names, params)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Récupérer les noms des 10 classes (dans l'ordre alphabétique de Keras)
class_names = test_ds.class_names

# 2. Prédire sur tout le jeu de test
predictions = cnn_model.predict(test_ds)
predicted_indices = np.argmax(predictions, axis=1)
confidences = np.max(predictions, axis=1)

# 3. Extraire toutes les images et les vraies étiquettes depuis test_ds
test_images = np.concatenate([images.numpy() for images, _ in test_ds], axis=0)
true_indices = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)

# 4. Afficher les 50 images dans une grille (5 lignes x 10 colonnes)
fig, axes = plt.subplots(5, 10, figsize=(22, 12))
axes = axes.flatten()

for i in range(len(test_images)):
    ax = axes[i]
    # .astype("uint8") est nécessaire car image_dataset_from_directory charge les pixels en float32 [0, 255]
    ax.imshow(test_images[i].astype("uint8"))
    
    pred_label = class_names[predicted_indices[i]]
    true_label = class_names[true_indices[i]]
    conf = confidences[i] * 100
    
    is_correct = (predicted_indices[i] == true_indices[i])
    color = "green" if is_correct else "red"
    
    ax.set_title(f"Prédit: {pred_label}\n(Vrai: {true_label})\n{conf:.0f}%", color=color, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()